Set up view on crop conditions, crop progress, and find the nass report data closest date to wasde report in aug

In [82]:
from pathlib import Path
import pandas as pd
import numpy as np

Load File Paths

In [83]:
# Project root
BASE_DIR = Path.cwd().parent

# Data files
CC_FILE = BASE_DIR / "data" / "nass" / "crop_condition.csv"


WASDE_FILE = (
    BASE_DIR
    / "data"
    / "wasde"
    / "wasde_processed_all_historicals.csv"
)

In [84]:
df = pd.read_csv(CC_FILE)
df_wasde = pd.read_csv(WASDE_FILE)


Crop Conditions View

In [85]:
df_cc = df.copy()

df_cc["Month"] = (
    pd.to_datetime(df_cc["Week Ending"]).dt.month
)

df_cc["Category"] = (
    df_cc["Data Item"]
    .str.extract(r"MEASURED IN PCT (.+)$")[0]
    .str.strip()
)

df_cc["Type"] = (
    df_cc["Data Item"]
    .str.split(" - ", n=1)
    .str[0]
    .str.strip()
)

aug_filtered = df_cc[
    (df_cc["Type"] == "CORN")&
    (df_cc["Month"] == 8)
    ]


aug_filtered[
    [
        "Year",
        "Month",
        "Period",
        "Week Ending",
        "Commodity",
        "Data Item",
        "Category",
        "Type",
        "Value"
    ]
]

,Year,Month,Period,Week Ending,Commodity,Data Item,Category,Type,Value
290,2026,8,WEEK #31,2026-08-02,CORN,"CORN - CONDITION, MEASURED IN PCT EXCELLENT",EXCELLENT,CORN,13
291,2026,8,WEEK #31,2026-08-02,CORN,"CORN - CONDITION, MEASURED IN PCT FAIR",FAIR,CORN,25
292,2026,8,WEEK #31,2026-08-02,CORN,"CORN - CONDITION, MEASURED IN PCT GOOD",GOOD,CORN,48
293,2026,8,WEEK #31,2026-08-02,CORN,"CORN - CONDITION, MEASURED IN PCT POOR",POOR,CORN,10
294,2026,8,WEEK #31,2026-08-02,CORN,"CORN - CONDITION, MEASURED IN PCT VERY POOR",VERY POOR,CORN,4
...,...,...,...,...,...,...,...,...,...
8696,2010,8,WEEK #34,2010-08-29,CORN,"CORN - CONDITION, MEASURED IN PCT EXCELLENT",EXCELLENT,CORN,22
8697,2010,8,WEEK #34,2010-08-29,CORN,"CORN - CONDITION, MEASURED IN PCT FAIR",FAIR,CORN,20
8698,2010,8,WEEK #34,2010-08-29,CORN,"CORN - CONDITION, MEASURED IN PCT GOOD",GOOD,CORN,48
8699,2010,8,WEEK #34,2010-08-29,CORN,"CORN - CONDITION, MEASURED IN PCT POOR",POOR,CORN,7


In [86]:
crop_condition_pivot = aug_filtered.pivot(
    index="Week Ending",
    columns="Category",
    values="Value"
)[
    ["EXCELLENT", "GOOD", "FAIR", "POOR", "VERY POOR"]
]

crop_condition_pivot

Category,EXCELLENT,GOOD,FAIR,POOR,VERY POOR
Week Ending,,,,,
2010-08-01,24,47,19,7,3
2010-08-08,23,48,19,7,3
2010-08-15,23,46,20,8,3
2010-08-22,23,47,20,7,3
2010-08-29,22,48,20,7,3
...,...,...,...,...,...
2026-08-02,13,48,25,10,4
2026-08-09,13,48,25,10,4
2026-08-16,13,47,25,10,5


USDA style crop conditions

In [97]:
weekly_filtered = df_cc[
    (df_cc["Type"] == "CORN")
    ]

weekly_filtered["Week"] = (
    weekly_filtered["Period"]
    .str.extract(r"WEEK #(\d+)")[0]
    .astype(int)
)
weekly_filtered["Good_Excellent"] = (
    aug_filtered["Category"].isin(["GOOD", "EXCELLENT"])
)

ge = (
    weekly_filtered[weekly_filtered["Category"].isin(["GOOD", "EXCELLENT"])]
    .groupby(["Week", "Year"])["Value"]
    .sum()
    .reset_index()
)

ge_table = ge.pivot(
    index="Week",
    columns="Year",
    values="Value"
)

ge_table

Year,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
Week,,,,,,,,,,,,,,,,,
19,67.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,71.0,NaN,77.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,76.0,63.0,72.0,NaN,NaN,74.0,72.0,65.0,79.0,NaN,70.0,76.0,NaN,69.0,NaN,68.0,NaN
22,76.0,67.0,72.0,63.0,76.0,74.0,75.0,68.0,78.0,NaN,74.0,72.0,73.0,64.0,75.0,69.0,67.0
23,77.0,69.0,66.0,63.0,75.0,74.0,75.0,67.0,77.0,59.0,75.0,68.0,72.0,61.0,74.0,71.0,67.0
24,75.0,70.0,63.0,64.0,76.0,73.0,75.0,67.0,78.0,59.0,71.0,65.0,70.0,55.0,72.0,72.0,68.0
25,73.0,68.0,56.0,65.0,74.0,71.0,75.0,67.0,77.0,56.0,72.0,64.0,67.0,50.0,69.0,70.0,68.0
26,71.0,69.0,48.0,67.0,75.0,68.0,75.0,68.0,76.0,56.0,73.0,64.0,64.0,51.0,67.0,73.0,67.0
27,73.0,69.0,40.0,68.0,75.0,69.0,76.0,65.0,75.0,57.0,71.0,65.0,64.0,55.0,68.0,74.0,67.0


In [98]:
display_table = ge_table.copy()

display_table = display_table.map(
    lambda x: "-" if pd.isna(x) else f"{x:.0f}%"
)

In [99]:
display_table

Year,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
Week,,,,,,,,,,,,,,,,,
19,67%,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
20,71%,-,77%,-,-,-,-,-,-,-,-,-,-,-,-,-,-
21,76%,63%,72%,-,-,74%,72%,65%,79%,-,70%,76%,-,69%,-,68%,-
22,76%,67%,72%,63%,76%,74%,75%,68%,78%,-,74%,72%,73%,64%,75%,69%,67%
23,77%,69%,66%,63%,75%,74%,75%,67%,77%,59%,75%,68%,72%,61%,74%,71%,67%
24,75%,70%,63%,64%,76%,73%,75%,67%,78%,59%,71%,65%,70%,55%,72%,72%,68%
25,73%,68%,56%,65%,74%,71%,75%,67%,77%,56%,72%,64%,67%,50%,69%,70%,68%
26,71%,69%,48%,67%,75%,68%,75%,68%,76%,56%,73%,64%,64%,51%,67%,73%,67%
27,73%,69%,40%,68%,75%,69%,76%,65%,75%,57%,71%,65%,64%,55%,68%,74%,67%


SyntaxError: invalid syntax (3758736981.py, line 1)